# Actividad 1: Obtener y preparar datos desde Hugging Face

Este notebook coordina el proceso de descarga de un dataset desde Hugging Face, la aplicación de transformaciones de datos utilizando pandas, y la exportación de un resumen estadístico y estructural a un archivo `.txt`.

In [1]:
# Instalación de dependencias necesarias
!pip install datasets pandas

## 1. Creación de las Clases en Python

Creamos el archivo `hf_loader.py` que contiene la clase encargada de conectarse a Hugging Face y extraer el dataset como un DataFrame de Pandas.

In [2]:
%%writefile hf_loader.py
import pandas as pd
from datasets import load_dataset

class HFDatasetLoader:
    def __init__(self, dataset_name: str, split: str = "train"):
        self.dataset_name = dataset_name
        self.split = split

    def load_to_dataframe(self) -> pd.DataFrame:
        """Carga el dataset desde Hugging Face y lo convierte en DataFrame de pandas."""
        dataset = load_dataset(self.dataset_name, split=self.split)
        return dataset.to_pandas()

Overwriting hf_loader.py


## 2. Creamos el archivo `dataset_summarizer.py` que contiene la clase encargada de analizar un DataFrame y generar un reporte de resumen en formato `.txt`.

In [3]:
%%writefile dataset_summarizer.py
import pandas as pd
import io

class DatasetSummarizer:
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def generate_summary(self, output_filename: str = "resumen_dataset.txt"):
        """Genera un archivo .txt con la estructura, tipos de datos, nulos y estadísticas."""
        buffer = io.StringIO()
        self.df.info(buf=buffer)
        info_str = buffer.getvalue()

        summary_content = []
        summary_content.append("=" * 50)
        summary_content.append("RESUMEN DEL DATAFRAME TRANSFORMADO")
        summary_content.append("=" * 50)
        summary_content.append(f"\nDimensiones: {self.df.shape[0]} filas, {self.df.shape[1]} columnas\n")

        summary_content.append("--- INFORMACIÓN GENERAL Y TIPOS DE DATOS ---")
        summary_content.append(info_str)

        summary_content.append("\n--- VALORES NULOS POR COLUMNA ---")
        summary_content.append(str(self.df.isnull().sum()))

        summary_content.append("\n--- ESTADÍSTICAS DESCRIPTIVAS ---")
        summary_content.append(str(self.df.describe(include='all')))

        with open(output_filename, "w", encoding="utf-8") as f:
            f.write("\n".join(summary_content))

        print(f"Resumen guardado exitosamente en '{output_filename}'")

Overwriting dataset_summarizer.py


## Ejecución, Carga y Orquestación del Flujo

In [4]:
# Creamos el archivo `dataset_summarizer.py` que contiene la clase encargada de analizar un DataFrame y generar un reporte de resumen en formato `.txt`.
from hf_loader import HFDatasetLoader

# Cargamos el dataset utilizando la clase HFDatasetLoader
loader = HFDatasetLoader("scikit-learn/iris", split="train")
df_original = loader.load_to_dataframe()

# Mostramos las primeras filas originales
print(type(df_original), df_original.shape)
display(df_original.head())

<class 'pandas.core.frame.DataFrame'> (150, 6)


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## 4. Aplicación de 4 Transformaciones Distintas

In [5]:
# Copiamos el dataframe para no modificar el original directamente
df_transformed = df_original.copy()

# --- Transformación 1: Renombrar columnas a español para mayor claridad ---
# Usamos los nombres reales de las columnas del dataset 'scikit-learn/iris' cargado
column_mapping = {
    'SepalLengthCm': 'largo_sepalo_cm',
    'SepalWidthCm': 'ancho_sepalo_cm',
    'PetalLengthCm': 'largo_petalo_cm',
    'PetalWidthCm': 'ancho_petalo_cm',
    'Species': 'especie'
}
df_transformed = df_transformed.rename(columns=column_mapping)
print("1. Columnas renombradas:")
display(df_transformed.head(2))

# --- Transformación 2: Filtrar filas basadas en una condición ---
# Filtramos para quedarnos únicamente con registros que tengan un largo de sépalo mayor a 5.0 cm
df_transformed = df_transformed[df_transformed['largo_sepalo_cm'] > 5.0].copy()
print(f"2. DataFrame filtrado (largo_sepalo_cm > 5.0). Nuevas dimensiones: {df_transformed.shape}")

# --- Transformación 3: Mapear valores de la especie a nombres más limpios ---
# Las especies originales vienen como 'Iris-setosa', 'Iris-versicolor', 'Iris-virginica'.
# Las mapeamos a nombres más simples/cortos.
species_map = {
    'Iris-setosa': 'setosa',
    'Iris-versicolor': 'versicolor',
    'Iris-virginica': 'virginica'
}
df_transformed['especie'] = df_transformed['especie'].map(species_map)
print("3. Mapeo de especies aplicado:")
display(df_transformed.head(2))

# --- Transformación 4: Crear una nueva columna calculada ---
# Calculamos la proporción/ratio entre el largo y ancho del pétalo
df_transformed['ratio_petalo'] = df_transformed['largo_petalo_cm'] / df_transformed['ancho_petalo_cm']
print("4. Columna calculada 'ratio_petalo' agregada:")
display(df_transformed.head(5))

1. Columnas renombradas:


,Id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa


2. DataFrame filtrado (largo_sepalo_cm > 5.0). Nuevas dimensiones: (118, 6)
3. Mapeo de especies aplicado:


,Id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,1,5.1,3.5,1.4,0.2,setosa
5,6,5.4,3.9,1.7,0.4,setosa


4. Columna calculada 'ratio_petalo' agregada:


,Id,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,ratio_petalo
0,1,5.1,3.5,1.4,0.2,setosa,7.00
5,6,5.4,3.9,1.7,0.4,setosa,4.25
10,11,5.4,3.7,1.5,0.2,setosa,7.50
14,15,5.8,4.0,1.2,0.2,setosa,6.00
15,16,5.7,4.4,1.5,0.4,setosa,3.75


##  Resumen y Exportación

Finalmente, enviamos el DataFrame transformado a la clase `DatasetSummarizer` para generar el reporte de texto final.

In [6]:
# Instanciar el resumidor con el dataframe resultante
from dataset_summarizer import DatasetSummarizer
summarizer = DatasetSummarizer(df_transformed)

# Generar el archivo txt de salida
summarizer.generate_summary("resumen_dataset.txt")
with open("resumen_dataset.txt", "r", encoding="utf-8") as file:
    print(file.read())


Resumen guardado exitosamente en 'resumen_dataset.txt'
RESUMEN DEL DATAFRAME TRANSFORMADO

Dimensiones: 118 filas, 7 columnas

--- INFORMACIÓN GENERAL Y TIPOS DE DATOS ---
<class 'pandas.core.frame.DataFrame'>
Index: 118 entries, 0 to 149
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Id               118 non-null    int64  
 1   largo_sepalo_cm  118 non-null    float64
 2   ancho_sepalo_cm  118 non-null    float64
 3   largo_petalo_cm  118 non-null    float64
 4   ancho_petalo_cm  118 non-null    float64
 5   especie          118 non-null    object 
 6   ratio_petalo     118 non-null    float64
dtypes: float64(5), int64(1), object(1)
memory usage: 7.4+ KB


--- VALORES NULOS POR COLUMNA ---
Id                 0
largo_sepalo_cm    0
ancho_sepalo_cm    0
largo_petalo_cm    0
ancho_petalo_cm    0
especie            0
ratio_petalo       0
dtype: int64

--- ESTADÍSTICAS DESCRIPTIVAS ---
                Id  la

### Transformaciones aplicadas
1. **Renombrar columnas:** se tradujeron los nombres de las columnas al español.
2. **Filtrar filas:** se conservaron solo los registros con largo de sépalo mayor a 5.0 cm (de 150 a 118 filas).
3. **Mapear valores:** se simplificaron los nombres de especie (`Iris-setosa` → `setosa`, etc.).
4. **Columna calculada:** se creó `ratio_petalo` = largo del pétalo / ancho del pétalo.